## Model final dan evaluasi test

Notebook ini melatih model final, membuka test set **satu kali**, lalu menyimpan artifact (`models/model.joblib` dan `models/metadata.json`).

- Model: HistGradientBoostingRegressor (`random_state=42`), fitur dari `src/features.py`
- Data latih: Juli 2024 - April 2026 (train + validation), 79.015 baris
- Data uji: Mei - Juni 2026 (test), 7.205 baris, dibuka satu kali
- Batas alarm peringatan: 45 µg/m³, dipilih di validation (F1 tertinggi), bukan di test

In [1]:
import sys, json; sys.path.append("..")
import pandas as pd, joblib, sklearn, numpy
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from src.features import buat_lag, tambah_fitur, FITUR

BATAS_ALARM = 45
df = pd.read_csv("../data/clean.csv", parse_dates=["waktu"])
df["target"] = df.groupby("stasiun")["pm25_ugm3"].shift(-3)
d = buat_lag(df).dropna(subset=["pm25_ugm3", "target"])
X = tambah_fitur(d)

latih = d["waktu"] < "2026-05-01"    # train + validation
uji = d["waktu"] >= "2026-05-01"     # test: dibuka SEKALI di cell berikutnya
model = HistGradientBoostingRegressor(random_state=42).fit(X[latih], d.loc[latih, "target"])
print(latih.sum(), uji.sum())

79015 7205


In [2]:
p, y = model.predict(X[uji]), d.loc[uji, "target"]
sekarang = d.loc[uji, "pm25_ugm3"]
kej = y > 55
a = p > BATAS_ALARM
tp = int((kej & a).sum())

hasil = {
    "mae_model": round(mean_absolute_error(y, p), 3),
    "mae_baseline": round(mean_absolute_error(y, sekarang), 3),
    "jam_di_atas_55": int(kej.sum()),
    "alarm": int(a.sum()),
    "recall": round(tp / max(kej.sum(), 1), 3),
    "precision": round(tp / max(a.sum(), 1), 3),
    "recall_baseline": round(int((kej & (sekarang > 55)).sum()) / max(kej.sum(), 1), 3),
}
hasil

{'mae_model': 4.668,
 'mae_baseline': 6.5,
 'jam_di_atas_55': 239,
 'alarm': 517,
 'recall': np.float64(0.736),
 'precision': np.float64(0.34),
 'recall_baseline': np.float64(0.301)}

### Hasil test (Mei - Juni 2026)

| | Model | Baseline (persistence) |
|---|---|---|
| MAE (µg/m³) | **4,668** | 6,500 |
| Recall jam >55 | **0,736** | 0,301 |

- Jam dengan PM2.5 >55 µg/m³ di test: 239 jam. Model membunyikan 517 alarm (prediksi >45), dengan recall 0,736 dan precision 0,340, artinya sekitar 176 dari 239 kejadian tertangkap, sedangkan sekitar dua dari tiga alarm adalah alarm palsu.
- Dibanding validation (MAE 4,695; recall 0,465; precision 0,328), MAE hampir sama. Recall test lebih tinggi, tetapi validation hanya punya 43 jam kejadian, jadi perbandingan recall tidak dijadikan klaim bahwa model membaik.
- Jumlah jam >55 jauh lebih banyak di test (239 vs 43) karena test jatuh di awal musim kemarau.
- Keterbatasan: precision rendah (banyak alarm palsu) sebagai harga batas alarm 45 yang dipilih agar lebih waspada. Batas antar-periode tidak diberi jeda 3 jam (dampaknya sangat kecil, sekitar 15 baris latih).
- Setelah test dibuka, model dan batas alarm tidak diubah.

In [3]:
import os; os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/model.joblib")

metadata = {
    "versi_model": "1.0.0",
    "algoritma": "HistGradientBoostingRegressor(random_state=42)",
    "target": "pm25_ugm3 tiga jam ke depan",
    "fitur_input": FITUR,
    "stasiun": ["Antapani", "Cibeureum", "Dago", "Soekarno-Hatta", "Ujungberung"],
    "batas_alarm": BATAS_ALARM,
    "data_latih": "Juli 2024 - April 2026 (train + validation)",
    "metrik_validation": {"mae_model": 4.695, "mae_baseline": 6.255, "recall": 0.465, "precision": 0.328},
    "metrik_test": hasil,
    "versi_library": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__,
                      "pandas": pd.__version__, "numpy": numpy.__version__},
}
json.dump(metadata, open("../models/metadata.json", "w"), indent=2)